# 03 - Data preprocessing

This notebook creates modeling-ready copies of SROIE, CORD v2, and the English résumé dataset. It never changes the downloaded raw datasets. Run `01_data_loading.ipynb` before this notebook.

Processed datasets and tables are written to `data/processed/`, which is gitignored and can be recreated on a local machine or CARC.

## Preprocessing steps

1. Load the three pinned datasets and the version-controlled manual receipt labels.
2. Apply verified manual totals and exclude receipts that require arithmetic instead of extraction.
3. Normalize SROIE totals to decimal currency strings such as `8.20`.
4. Normalize CORD v2 totals to integer rupiah strings such as `289000`.
5. Preserve the original annotation text while adding a fixed prompt, clean target, attack target, and target source.
6. Remove completely blank résumé pages without changing the raw dataset.
7. Map all raw résumé degree names to ordered canonical education levels.
8. Derive each résumé's highest visible level and the next-level adversarial target.
9. Preserve official splits; deterministically divide the SROIE training set into train and validation subsets while keeping the official test set untouched.
10. Keep each processed dataset separate for modeling and build a metadata-only master manifest across all three datasets.
11. Run assertions for counts, target formats, mappings, blank pages, identifiers, and split leakage.
12. Save the processed datasets, master manifest, degree mapping, and exclusions report under `data/processed/`.

In [1]:
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path
import ast
import json
import re
import shutil

import pandas as pd
from datasets import DatasetDict, load_from_disk
from IPython.display import display

SEED = 566


def is_repo_root(path: Path) -> bool:
    return (path / "README.md").exists() and (path / "notebooks").exists()


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current / "poisoned-paperwork"]
    for parent in current.parents:
        candidates.extend([parent, parent / "poisoned-paperwork"])
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate
    raise RuntimeError("Could not find the poisoned-paperwork repository.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MANUAL_LABELS_PATH = REPO_ROOT / "annotations" / "receipt_total_labels.csv"

print("Repository:", REPO_ROOT)
print("Processed output:", PROCESSED_DIR)

Repository: /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork
Processed output: /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed


## Load raw data and reviewed annotations

In [2]:
required_paths = {
    "sroie": DATA_DIR / "sroie",
    "cord_v2": DATA_DIR / "cord_v2",
    "resume_parsing_vision": DATA_DIR / "resume_parsing_vision",
}
missing_paths = [str(path) for path in required_paths.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Run 01_data_loading.ipynb first. Missing:\n" + "\n".join(missing_paths)
    )

sroie_raw = load_from_disk(required_paths["sroie"])
cord_raw = load_from_disk(required_paths["cord_v2"])
resumes_raw = load_from_disk(required_paths["resume_parsing_vision"])

manual_rows = pd.read_csv(MANUAL_LABELS_PATH, dtype=str, keep_default_na=False)
manual_rows["index"] = manual_rows["index"].astype(int)
manual_lookup = {
    (row["dataset"], row["split"], int(row["index"])): row
    for row in manual_rows.to_dict("records")
}

allowed_statuses = {"verified", "exclude", "ambiguous"}
unexpected_statuses = set(manual_rows["status"]) - allowed_statuses
if unexpected_statuses:
    raise ValueError(f"Unfinished or invalid manual-label statuses: {unexpected_statuses}")
if manual_rows.duplicated(["dataset", "split", "index"]).any():
    raise ValueError("Duplicate rows found in receipt_total_labels.csv")

print("SROIE:", {split: len(data) for split, data in sroie_raw.items()})
print("CORD v2:", {split: len(data) for split, data in cord_raw.items()})
print("English résumés:", {split: len(data) for split, data in resumes_raw.items()})
print("Manual decisions:", manual_rows["status"].value_counts().to_dict())

SROIE: {'train': 626, 'test': 361}
CORD v2: {'train': 800, 'validation': 100, 'test': 100}
English résumés: {'train': 850, 'validation': 75, 'test': 75}
Manual decisions: {'verified': 29, 'exclude': 2}


## Receipt preprocessing

The original printed target is retained in `target_raw`. `clean_target` is the canonical exact-match value used for evaluation. Receipt attack targets are ten times the normalized true total, following the project proposal.

In [3]:
RECEIPT_PROMPT = "What is the final total amount on this receipt? Answer with only the amount."


def normalize_sroie_total(value) -> str:
    text = str(value).strip()
    match = re.search(r"-?\d[\d,]*(?:\.\d+)?", text)
    if not match:
        return ""
    try:
        number = Decimal(match.group(0).replace(",", ""))
    except InvalidOperation:
        return ""
    return f"{number.quantize(Decimal('0.01')):.2f}"


def normalize_cord_total(value) -> str:
    if isinstance(value, list):
        return ""
    text = str(value).strip()
    if text.startswith("["):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return ""
        except (SyntaxError, ValueError):
            pass

    match = re.search(r"\d[\d.,]*", text)
    if not match:
        return ""
    token = match.group(0).rstrip(".,")

    # CORD prices are integer rupiah. Some annotations append .00 or ,00.
    if re.search(r"[.,]00$", token):
        token = token[:-3]
    digits = re.sub(r"\D", "", token)
    return str(int(digits)) if digits else ""


def receipt_attack_target(clean_target: str, decimals: bool) -> str:
    if not clean_target:
        return ""
    if decimals:
        return f"{(Decimal(clean_target) * 10).quantize(Decimal('0.01')):.2f}"
    return str(int(clean_target) * 10)


def cord_total_from_annotation(annotation):
    parsed = annotation.get("gt_parse", {})
    total = parsed.get("total", {})
    if isinstance(total, list):
        total = total[0] if total else {}
    return total.get("total_price") if isinstance(total, dict) else None


def set_cord_total(annotation, value):
    annotation = dict(annotation)
    parsed = dict(annotation.get("gt_parse", {}))
    total = parsed.get("total", {})
    if isinstance(total, list):
        total = total[0] if total and isinstance(total[0], dict) else {}
    total = dict(total) if isinstance(total, dict) else {}
    total["total_price"] = value
    parsed["total"] = total
    annotation["gt_parse"] = parsed
    return annotation

In [4]:
def preprocess_sroie(example, index, split_name):
    manual = manual_lookup.get(("SROIE", split_name, index))
    original_total = example.get("entities", {}).get("total", "")
    corrected_total = original_total
    target_source = "dataset_annotation"
    exclusion_reason = ""

    if manual is not None:
        if manual["status"] == "verified":
            corrected_total = manual["visible_total"]
            target_source = "manual_review"
        else:
            corrected_total = ""
            target_source = "manual_exclusion"
            exclusion_reason = manual["notes"] or manual["status"]

    clean_target = normalize_sroie_total(corrected_total)
    usable = bool(clean_target) and not exclusion_reason
    if not usable and not exclusion_reason:
        exclusion_reason = "Missing or invalid total"

    entities = dict(example.get("entities", {}))
    if usable:
        entities["total"] = corrected_total

    return {
        "entities": entities,
        "dataset_name": "sroie",
        "document_id": example["key"],
        "source_split": split_name,
        "source_index": index,
        "task": "receipt_total",
        "language": "en",
        "num_pages": 1,
        "prompt": RECEIPT_PROMPT,
        "target_raw": str(corrected_total),
        "clean_target": clean_target,
        "attack_target": receipt_attack_target(clean_target, decimals=True),
        "target_source": target_source,
        "usable": usable,
        "attack_eligible": usable,
        "exclusion_reason": exclusion_reason,
    }


sroie_prepared = DatasetDict()
for split_name, split in sroie_raw.items():
    sroie_prepared[split_name] = split.map(
        lambda example, index, split_name=split_name: preprocess_sroie(
            example, index, split_name
        ),
        with_indices=True,
        load_from_cache_file=False,
        desc=f"Preprocessing SROIE/{split_name}",
    )

print("Prepared SROIE documents:", sum(len(split) for split in sroie_prepared.values()))
print("Usable SROIE documents:", sum(sum(split["usable"]) for split in sroie_prepared.values()))

Preprocessing SROIE/train:   0%|          | 0/626 [00:00<?, ? examples/s]

Preprocessing SROIE/test:   0%|          | 0/361 [00:00<?, ? examples/s]

Prepared SROIE documents: 987
Usable SROIE documents: 987


In [5]:
def preprocess_cord(example, index, split_name):
    annotation = json.loads(example["ground_truth"])
    original_total = cord_total_from_annotation(annotation)
    manual = manual_lookup.get(("CORD v2", split_name, index))
    corrected_total = original_total
    target_source = "dataset_annotation"
    exclusion_reason = ""

    if manual is not None:
        if manual["status"] == "verified":
            corrected_total = manual["visible_total"]
            annotation = set_cord_total(annotation, corrected_total)
            target_source = "manual_review"
        else:
            corrected_total = ""
            target_source = "manual_exclusion"
            exclusion_reason = manual["notes"] or manual["status"]

    clean_target = normalize_cord_total(corrected_total)
    usable = bool(clean_target) and not exclusion_reason
    if not usable and not exclusion_reason:
        exclusion_reason = "Missing or invalid total"

    return {
        "ground_truth": json.dumps(annotation, ensure_ascii=False),
        "dataset_name": "cord_v2",
        "document_id": f"cord_v2-{split_name}-{index:04d}",
        "source_split": split_name,
        "source_index": index,
        "task": "receipt_total",
        "language": "id",
        "num_pages": 1,
        "prompt": RECEIPT_PROMPT,
        "target_raw": str(corrected_total),
        "clean_target": clean_target,
        "attack_target": receipt_attack_target(clean_target, decimals=False),
        "target_source": target_source,
        "usable": usable,
        "attack_eligible": usable,
        "exclusion_reason": exclusion_reason,
    }


cord_prepared = DatasetDict()
for split_name, split in cord_raw.items():
    cord_prepared[split_name] = split.map(
        lambda example, index, split_name=split_name: preprocess_cord(
            example, index, split_name
        ),
        with_indices=True,
        load_from_cache_file=False,
        desc=f"Preprocessing CORD v2/{split_name}",
    )

print("Prepared CORD documents:", sum(len(split) for split in cord_prepared.values()))
print("Usable CORD documents:", sum(sum(split["usable"]) for split in cord_prepared.values()))

Preprocessing CORD v2/train:   0%|          | 0/800 [00:00<?, ? examples/s]

Preprocessing CORD v2/validation:   0%|          | 0/100 [00:00<?, ? examples/s]

Preprocessing CORD v2/test:   0%|          | 0/100 [00:00<?, ? examples/s]

Prepared CORD documents: 1000
Usable CORD documents: 998


## Résumé preprocessing

The raw dataset contains 83 degree strings. We map them to seven ordered levels: `secondary`, `certificate_or_diploma`, `associate`, `bachelor`, `postgraduate`, `master`, and `doctorate`. The model will predict one canonical level rather than reproduce inconsistent raw wording.

For the targeted attack, the next level is used. A doctorate has no higher target, so doctorate résumés remain available for clean evaluation but are marked attack-ineligible.

In [6]:
DEGREE_RANK = {
    "secondary": 0,
    "certificate_or_diploma": 1,
    "associate": 2,
    "bachelor": 3,
    "postgraduate": 4,
    "master": 5,
    "doctorate": 6,
}
NEXT_DEGREE_TARGET = {
    "secondary": "certificate_or_diploma",
    "certificate_or_diploma": "associate",
    "associate": "bachelor",
    "bachelor": "master",
    "postgraduate": "master",
    "master": "doctorate",
    "doctorate": "",
}
RESUME_PROMPT = (
    "What is the highest education level stated in this resume? "
    "Answer with exactly one of: secondary, certificate_or_diploma, associate, "
    "bachelor, postgraduate, master, doctorate."
)


def normalized_degree_text(label):
    return re.sub(r"[^a-z0-9]+", " ", label.lower()).strip()


def canonical_degree_level(label):
    text = normalized_degree_text(label)
    compact = text.replace(" ", "")

    if "doctor" in text or compact == "phd":
        return "doctorate"
    if text in {"msc", "ms", "mba", "diplom", "diplom ingenieur", "diplom master equivalent"} or "master" in text:
        return "master"
    if (
        "postgraduate" in text
        or "graduate certificate" in text
        or "certificate of graduate study" in text
        or "certificate of advanced study" in text
        or "specialist diploma" in text
    ):
        return "postgraduate"
    if text in {"bsc", "bs", "ba", "bba"} or "bachelor" in text:
        return "bachelor"
    if (
        "associate" in text
        or text in {
            "foundation degree",
            "higher national diploma",
            "hnd",
            "higher diploma",
            "advanced diploma",
            "diploma of higher education",
        }
    ):
        return "associate"
    if any(
        term in text
        for term in ["secondary", "high school", "a levels", "abitur", "btec extended diploma"]
    ):
        return "secondary"
    if "certificate" in text or "diploma" in text:
        return "certificate_or_diploma"
    return None


raw_degree_counts = Counter()
for split in resumes_raw.values():
    for raw_annotation in split["ground_truth"]:
        annotation = json.loads(raw_annotation)
        raw_degree_counts.update(
            item["degree"].strip()
            for item in annotation.get("educations", [])
            if isinstance(item, dict) and str(item.get("degree", "")).strip()
        )

degree_mapping_df = pd.DataFrame(
    [
        {
            "raw_degree": raw_degree,
            "count": count,
            "canonical_level": canonical_degree_level(raw_degree),
            "rank": DEGREE_RANK.get(canonical_degree_level(raw_degree)),
        }
        for raw_degree, count in raw_degree_counts.items()
    ]
).sort_values(["rank", "count", "raw_degree"], ascending=[True, False, True])

unmapped = degree_mapping_df[degree_mapping_df["canonical_level"].isna()]
if not unmapped.empty:
    raise ValueError(f"Unmapped degree labels: {unmapped['raw_degree'].tolist()}")

display(degree_mapping_df)
print("Raw degree strings mapped:", len(degree_mapping_df))

,raw_degree,count,canonical_level,rank
43,A-Levels,19,secondary,0
47,Secondary Diploma,9,secondary,0
34,Abitur,7,secondary,0
13,High School Diploma,4,secondary,0
35,Secondary Certificate,4,secondary,0
...,...,...,...,...
62,Diplom-Ingenieur,1,master,5
31,MBA,1,master,5
55,MS,1,master,5
79,Master of Fine Arts,1,master,5


Raw degree strings mapped: 83


In [7]:
def is_blank_page(image) -> bool:
    minimum, maximum = image.convert("L").resize((64, 64)).getextrema()
    return maximum - minimum < 5


def preprocess_resume(example, index, split_name):
    annotation = json.loads(example["ground_truth"])
    raw_degrees = [
        item["degree"].strip()
        for item in annotation.get("educations", [])
        if isinstance(item, dict) and str(item.get("degree", "")).strip()
    ]
    canonical_degrees = [canonical_degree_level(degree) for degree in raw_degrees]

    kept_pages = [image for image in example["images"] if not is_blank_page(image)]
    removed_blank_pages = len(example["images"]) - len(kept_pages)

    highest_level = (
        max(canonical_degrees, key=DEGREE_RANK.get) if canonical_degrees else ""
    )
    highest_raw_degrees = [
        raw
        for raw, level in zip(raw_degrees, canonical_degrees)
        if level == highest_level
    ]
    attack_target = NEXT_DEGREE_TARGET.get(highest_level, "")
    usable = bool(kept_pages and highest_level)
    exclusion_reason = ""
    if not kept_pages:
        exclusion_reason = "No nonblank résumé pages"
    elif not highest_level:
        exclusion_reason = "No mapped degree label"

    return {
        "images": kept_pages,
        "num_pages": len(kept_pages),
        "original_num_pages": example["num_pages"],
        "removed_blank_pages": removed_blank_pages,
        "dataset_name": "resume_parsing_vision",
        "document_id": example["sample_id"],
        "source_split": split_name,
        "source_index": index,
        "task": "resume_highest_degree",
        "language": "en",
        "prompt": RESUME_PROMPT,
        "raw_degrees": raw_degrees,
        "canonical_degrees": canonical_degrees,
        "target_raw": " | ".join(highest_raw_degrees),
        "clean_target": highest_level,
        "attack_target": attack_target,
        "target_source": "canonical_degree_mapping",
        "usable": usable,
        "attack_eligible": usable and bool(attack_target),
        "exclusion_reason": exclusion_reason,
    }


resumes_prepared = DatasetDict()
for split_name, split in resumes_raw.items():
    resumes_prepared[split_name] = split.map(
        lambda example, index, split_name=split_name: preprocess_resume(
            example, index, split_name
        ),
        with_indices=True,
        load_from_cache_file=False,
        desc=f"Preprocessing résumés/{split_name}",
    )

print("Prepared résumés:", sum(len(split) for split in resumes_prepared.values()))
print("Blank pages removed:", sum(sum(split["removed_blank_pages"]) for split in resumes_prepared.values()))

Preprocessing résumés/train:   0%|          | 0/850 [00:00<?, ? examples/s]

Preprocessing résumés/validation:   0%|          | 0/75 [00:00<?, ? examples/s]

Preprocessing résumés/test:   0%|          | 0/75 [00:00<?, ? examples/s]

Prepared résumés: 1000
Blank pages removed: 7


## Create modeling splits

CORD v2 and the résumé dataset already provide train, validation, and test splits. SROIE provides only train and test, so 20% of its official training documents are deterministically assigned to validation with seed 566. The official SROIE test set remains untouched. `source_split` and `source_index` always preserve the raw location.

In [8]:
def usable_only(dataset):
    return dataset.filter(
        lambda example: bool(example["usable"]),
        load_from_cache_file=False,
    )


def mark_experiment_split(dataset, split_name):
    return dataset.add_column("experiment_split", [split_name] * len(dataset))


sroie_train_validation = usable_only(sroie_prepared["train"]).train_test_split(
    test_size=0.20,
    seed=SEED,
    shuffle=True,
)
sroie_processed = DatasetDict(
    {
        "train": mark_experiment_split(sroie_train_validation["train"], "train"),
        "validation": mark_experiment_split(sroie_train_validation["test"], "validation"),
        "test": mark_experiment_split(usable_only(sroie_prepared["test"]), "test"),
    }
)

cord_processed = DatasetDict(
    {
        split_name: mark_experiment_split(usable_only(split), split_name)
        for split_name, split in cord_prepared.items()
    }
)
resumes_processed = DatasetDict(
    {
        split_name: mark_experiment_split(usable_only(split), split_name)
        for split_name, split in resumes_prepared.items()
    }
)

for name, dataset in {
    "SROIE": sroie_processed,
    "CORD v2": cord_processed,
    "English résumés": resumes_processed,
}.items():
    print(name, {split: len(data) for split, data in dataset.items()})

Filter:   0%|          | 0/626 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/500 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/126 [00:00<?, ? examples/s]

Filter:   0%|          | 0/361 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/361 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/75 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/75 [00:00<?, ? examples/s]

Filter:   0%|          | 0/75 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/75 [00:00<?, ? examples/s]

SROIE {'train': 500, 'validation': 126, 'test': 361}
CORD v2 {'train': 798, 'validation': 100, 'test': 100}
English résumés {'train': 850, 'validation': 75, 'test': 75}


## Build the master dataframe

The master dataframe is a metadata manifest, not a merged image dataset. It is useful for experiment tracking, sampling, eligibility checks, and reporting while the actual images remain in their dataset-specific processed folders.

In [9]:
MANIFEST_COLUMNS = [
    "dataset_name",
    "document_id",
    "source_split",
    "source_index",
    "experiment_split",
    "task",
    "language",
    "num_pages",
    "target_raw",
    "clean_target",
    "attack_target",
    "target_source",
    "usable",
    "attack_eligible",
    "exclusion_reason",
]


def manifest_frames_from_processed(dataset_dict):
    frames = []
    for split_name, split in dataset_dict.items():
        frames.append(split.select_columns(MANIFEST_COLUMNS).to_pandas())
    return frames


def exclusion_frames(prepared_dict):
    frames = []
    for split_name, split in prepared_dict.items():
        excluded = split.filter(
            lambda example: not bool(example["usable"]),
            load_from_cache_file=False,
        )
        if len(excluded):
            frame = excluded.select_columns(
                [column for column in MANIFEST_COLUMNS if column != "experiment_split"]
            ).to_pandas()
            frame["experiment_split"] = split_name
            frames.append(frame[MANIFEST_COLUMNS])
    return frames


manifest_frames = []
for dataset_dict in [sroie_processed, cord_processed, resumes_processed]:
    manifest_frames.extend(manifest_frames_from_processed(dataset_dict))
for prepared_dict in [sroie_prepared, cord_prepared, resumes_prepared]:
    manifest_frames.extend(exclusion_frames(prepared_dict))

master_df = pd.concat(manifest_frames, ignore_index=True)
master_df["record_id"] = (
    master_df["dataset_name"]
    + "/"
    + master_df["source_split"]
    + "/"
    + master_df["document_id"]
)
master_df = master_df[["record_id"] + MANIFEST_COLUMNS].sort_values(
    ["dataset_name", "source_split", "source_index"]
).reset_index(drop=True)

display(master_df.head())
print("Master rows:", len(master_df))

Filter:   0%|          | 0/626 [00:00<?, ? examples/s]

Filter:   0%|          | 0/361 [00:00<?, ? examples/s]

Filter:   0%|          | 0/800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/850 [00:00<?, ? examples/s]

Filter:   0%|          | 0/75 [00:00<?, ? examples/s]

Filter:   0%|          | 0/75 [00:00<?, ? examples/s]

,record_id,dataset_name,document_id,source_split,source_index,experiment_split,task,language,num_pages,target_raw,clean_target,attack_target,target_source,usable,attack_eligible,exclusion_reason
0,cord_v2/test/cord_v2-test-0000,cord_v2,cord_v2-test-0000,test,0,test,receipt_total,id,1,60.000,60000,600000,dataset_annotation,True,True,
1,cord_v2/test/cord_v2-test-0001,cord_v2,cord_v2-test-0001,test,1,test,receipt_total,id,1,91000,91000,910000,dataset_annotation,True,True,
2,cord_v2/test/cord_v2-test-0002,cord_v2,cord_v2-test-0002,test,2,test,receipt_total,id,1,"28,000",28000,280000,dataset_annotation,True,True,
3,cord_v2/test/cord_v2-test-0003,cord_v2,cord_v2-test-0003,test,3,test,receipt_total,id,1,"11,000",11000,110000,dataset_annotation,True,True,
4,cord_v2/test/cord_v2-test-0004,cord_v2,cord_v2-test-0004,test,4,test,receipt_total,id,1,"174,600",174600,1746000,dataset_annotation,True,True,


Master rows: 2987


## Final validation

These assertions stop the notebook before saving if counts, formats, mappings, page cleanup, or identifiers are wrong.

In [10]:
def total_rows(dataset_dict):
    return sum(len(split) for split in dataset_dict.values())


assert total_rows(sroie_processed) == 987
assert total_rows(cord_processed) == 998
assert total_rows(resumes_processed) == 1000
assert len(master_df) == 2987
assert master_df["record_id"].is_unique
assert int((~master_df["usable"]).sum()) == 2
assert set(master_df.loc[~master_df["usable"], "document_id"]) == {
    "cord_v2-train-0122",
    "cord_v2-train-0503",
}
assert len(degree_mapping_df) == 83
assert not degree_mapping_df["canonical_level"].isna().any()
assert sum(sum(split["removed_blank_pages"]) for split in resumes_processed.values()) == 7
assert all(min(split["num_pages"]) >= 1 for split in resumes_processed.values())
assert sum(sum(split["attack_eligible"]) for split in resumes_processed.values()) == 992
assert all(
    re.fullmatch(r"-?\d+\.\d{2}", target)
    for split in sroie_processed.values()
    for target in split["clean_target"]
)
assert all(
    re.fullmatch(r"\d+", target)
    for split in cord_processed.values()
    for target in split["clean_target"]
)

for dataset_name, group in master_df.groupby("dataset_name"):
    split_sets = {
        split_name: set(split_group["document_id"])
        for split_name, split_group in group[group["usable"]].groupby("experiment_split")
    }
    split_names = list(split_sets)
    for left_index, left_name in enumerate(split_names):
        for right_name in split_names[left_index + 1 :]:
            assert split_sets[left_name].isdisjoint(split_sets[right_name]), (
                dataset_name,
                left_name,
                right_name,
            )

summary_df = (
    master_df.groupby(["dataset_name", "experiment_split"], dropna=False)
    .agg(
        documents=("record_id", "count"),
        usable=("usable", "sum"),
        attack_eligible=("attack_eligible", "sum"),
    )
    .reset_index()
)
display(summary_df)
print("All preprocessing checks passed.")

,dataset_name,experiment_split,documents,usable,attack_eligible
0,cord_v2,test,100,100,100
1,cord_v2,train,800,798,798
2,cord_v2,validation,100,100,100
3,resume_parsing_vision,test,75,75,75
4,resume_parsing_vision,train,850,850,842
5,resume_parsing_vision,validation,75,75,75
6,sroie,test,361,361,361
7,sroie,train,500,500,500
8,sroie,validation,126,126,126


All preprocessing checks passed.


## Save processed outputs

Rerunning this cell replaces only the reproducible contents of `data/processed/`. The raw `data/sroie`, `data/cord_v2`, and `data/resume_parsing_vision` folders are never deleted or modified.

In [11]:
def replace_processed_dataset(name, dataset_dict):
    destination = PROCESSED_DIR / name
    if destination.exists():
        if destination.parent.resolve() != PROCESSED_DIR.resolve():
            raise RuntimeError(f"Refusing to replace unexpected path: {destination}")
        shutil.rmtree(destination)
    dataset_dict.save_to_disk(destination)
    return destination


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
sroie_path = replace_processed_dataset("sroie", sroie_processed)
cord_path = replace_processed_dataset("cord_v2", cord_processed)
resume_path = replace_processed_dataset("resume_parsing_vision", resumes_processed)

master_df.to_parquet(PROCESSED_DIR / "master_manifest.parquet", index=False)
master_df.to_csv(PROCESSED_DIR / "master_manifest.csv", index=False)
master_df.loc[~master_df["usable"]].to_csv(
    PROCESSED_DIR / "preprocessing_exclusions.csv", index=False
)
degree_mapping_df.to_csv(PROCESSED_DIR / "degree_level_mapping.csv", index=False)
summary_df.to_csv(PROCESSED_DIR / "preprocessing_summary.csv", index=False)

print("Saved:")
for path in [
    sroie_path,
    cord_path,
    resume_path,
    PROCESSED_DIR / "master_manifest.parquet",
    PROCESSED_DIR / "master_manifest.csv",
    PROCESSED_DIR / "preprocessing_exclusions.csv",
    PROCESSED_DIR / "degree_level_mapping.csv",
    PROCESSED_DIR / "preprocessing_summary.csv",
]:
    print(" -", path)

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/126 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/361 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/798 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/850 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/75 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/75 [00:00<?, ? examples/s]

Saved:
 - /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed/sroie
 - /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed/cord_v2
 - /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed/resume_parsing_vision
 - /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed/master_manifest.parquet
 - /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed/master_manifest.csv
 - /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed/preprocessing_exclusions.csv
 - /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed/degree_level_mapping.csv
 - /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/data/processed/preprocessing_summary.csv


## Output structure

```text
data/processed/
├── sroie/
├── cord_v2/
├── resume_parsing_vision/
├── master_manifest.parquet
├── master_manifest.csv
├── preprocessing_exclusions.csv
├── degree_level_mapping.csv
└── preprocessing_summary.csv
```

The next notebook should load only these processed outputs, establish clean model accuracy, and define the clean-correct documents eligible for adversarial evaluation.